# Step 3B — Synthetic Migration Dataset and Descriptive Analysis

This notebook implements the agreed **descriptive and exploratory** analysis for the legacy Android e-wallet Home screen migration scenario.

> **Synthetic-data disclosure:** every execution metric in this notebook—`execution_time_sec`, `token_usage`, `accuracy_score`, `test_pass_rate`, and `error_count`—is simulated teaching data. It is not collected from real Codex runs, real AI agents, production Android projects, or a validated benchmark. Any patterns, correlations, and outliers describe this synthetic sample only.

The notebook does not perform forecasting, machine learning, causal inference, or statistical generalization beyond the observed synthetic records.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Make the notebook work from the repository root or from analysis/.
candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    root for root in candidate_roots
    if (root / "data" / "migration_tasks.csv").exists()
)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from analysis.generate_synthetic_data import SEED as GENERATOR_SEED

SEED = 20260930
NUMERIC_METRICS = [
    "execution_time_sec",
    "token_usage",
    "accuracy_score",
    "test_pass_rate",
    "error_count",
]
CATEGORICAL_METRICS = [
    "agent_id",
    "status",
    "data_status",
]
plt.style.use("seaborn-v0_8-whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {platform.python_version()}")
for package in ("pandas", "numpy", "matplotlib"):
    print(f"{package}: {metadata.version(package)}")
assert GENERATOR_SEED == SEED
print(f"Fixed synthetic-data seed: {SEED}")
print("Synthetic observations are generated by analysis/generate_synthetic_data.py.")

## Load the raw inputs

The CSV files are kept separate: task metadata describes the migration work, while each observation is one simulated execution record. The raw inputs are not overwritten by this analysis.

In [ ]:
tasks = pd.read_csv(DATA_DIR / "migration_tasks.csv", dtype=str)
observations = pd.read_csv(DATA_DIR / "migration_observations.csv", dtype={
    "observation_id": str,
    "ticket_id": str,
    "agent_id": str,
    "execution_time_sec": float,
    "token_usage": int,
    "accuracy_score": float,
    "test_pass_rate": float,
    "error_count": int,
    "status": str,
    "data_status": str,
})

print(f"Tasks: {tasks.shape[0]:,} rows x {tasks.shape[1]} columns")
print(f"Observations: {observations.shape[0]:,} rows x {observations.shape[1]} columns")
display(tasks.head(3))
display(observations.head(3))

## Data-quality validation

The checks below validate required columns, nulls, duplicate identifiers, numeric ranges, categorical labels, and the task-to-observation foreign-key relationship.

In [ ]:
required_task_columns = {
    "ticket_id", "component_area", "migration_task",
    "migration_category", "difficulty",
}
required_observation_columns = {
    "observation_id", "ticket_id", "agent_id",
    *NUMERIC_METRICS, "status", "data_status",
}

assert required_task_columns.issubset(tasks.columns)
assert required_observation_columns.issubset(observations.columns)

null_summary = pd.concat(
    [
        tasks.isna().sum().rename("tasks_nulls"),
        observations.isna().sum().rename("observations_nulls"),
    ],
    axis=1,
).fillna(0).astype(int)
duplicate_summary = {
    "duplicate_task_ticket_ids": int(tasks["ticket_id"].duplicated().sum()),
    "duplicate_observation_ids": int(observations["observation_id"].duplicated().sum()),
}
foreign_key_gaps = sorted(set(observations["ticket_id"]) - set(tasks["ticket_id"]))
range_checks = {
    "non_negative_execution_time": bool((observations["execution_time_sec"] >= 0).all()),
    "non_negative_token_usage": bool((observations["token_usage"] >= 0).all()),
    "accuracy_in_0_1": bool(observations["accuracy_score"].between(0, 1).all()),
    "test_pass_rate_in_0_1": bool(observations["test_pass_rate"].between(0, 1).all()),
    "non_negative_error_count": bool((observations["error_count"] >= 0).all()),
}
allowed_categories = {
    "migration_category": {"Component", "Styling", "Behavior", "State"},
    "difficulty": {"Easy", "Medium", "Hard"},
    "status": {"Passed", "Passed with fixes", "Failed"},
    "data_status": {"synthetic"},
}
categorical_checks = {
    column: sorted(set(frame[column]) - allowed)
    for frame, column, allowed in [
        (tasks, "migration_category", allowed_categories["migration_category"]),
        (tasks, "difficulty", allowed_categories["difficulty"]),
        (observations, "status", allowed_categories["status"]),
        (observations, "data_status", allowed_categories["data_status"]),
    ]
}

assert not null_summary.any().any(), f"Unexpected nulls:\n{null_summary}"
assert all(value == 0 for value in duplicate_summary.values()), duplicate_summary
assert not foreign_key_gaps, foreign_key_gaps
assert all(range_checks.values()), range_checks
assert all(not invalid for invalid in categorical_checks.values()), categorical_checks
assert observations["data_status"].eq("synthetic").all()

validation_report = pd.DataFrame({
    "check": [
        "required schema",
        "null values",
        "duplicate IDs",
        "foreign-key coverage",
        "numeric ranges",
        "categorical labels",
        "synthetic disclosure label",
    ],
    "result": ["PASS"] * 7,
})
display(validation_report)
print("All validation checks passed.")

## Join task metadata to observations

The join is used only for descriptive grouping by component area, migration category, and difficulty. It does not turn the synthetic observations into real performance measurements.

In [ ]:
joined = observations.merge(
    tasks,
    on="ticket_id",
    how="left",
    validate="many_to_one",
)
assert joined.shape[0] == observations.shape[0]
display(joined[[
    "observation_id", "ticket_id", "component_area",
    "migration_category", "difficulty", "status", "data_status",
]].head())

## Frequency summaries for categorical variables

In [ ]:
for column in ["component_area", "migration_category", "difficulty", "agent_id", "status"]:
    frequency = (
        joined[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )
    display(Markdown(f"### {column}"))
    display(frequency)

## Descriptive statistics for synthetic numeric metrics

The summary reports count, mean, median (Q2), sample standard deviation, sample variance, minimum, Q1, Q2, Q3, maximum, and IQR. Values are descriptive summaries of this synthetic teaching dataset only.

In [ ]:
summary_rows = []
for variable in NUMERIC_METRICS:
    series = observations[variable].astype(float)
    q1, q2, q3 = series.quantile([0.25, 0.50, 0.75])
    summary_rows.append({
        "variable": variable,
        "count": int(series.count()),
        "mean": series.mean(),
        "median": series.median(),
        "std_dev": series.std(ddof=1),
        "variance": series.var(ddof=1),
        "minimum": series.min(),
        "q1": q1,
        "q2": q2,
        "q3": q3,
        "maximum": series.max(),
        "iqr": q3 - q1,
        "data_status": "synthetic",
        "metric_interpretation": "simulated teaching metric",
    })

descriptive_summary = pd.DataFrame(summary_rows)
descriptive_summary.to_csv(OUTPUT_DIR / "descriptive_summary.csv", index=False)
display(descriptive_summary.round(3))
print("Saved outputs/descriptive_summary.csv")

## Mean, median, and mode

Modes are reported for categorical variables and the clearly discrete `error_count`. Continuous metrics are not assigned a misleading unique mode; their values are better described by mean, median, quartiles, and IQR.

In [ ]:
mode_rows = []
for variable in ["component_area", "migration_category", "difficulty", "agent_id", "status", "error_count"]:
    modes = observations[variable].mode(dropna=True) if variable in observations else joined[variable].mode(dropna=True)
    mode_rows.append({
        "variable": variable,
        "mode": ", ".join(map(str, modes.tolist())) if len(modes) else "No useful unique mode",
        "mode_frequency": int(observations[variable].value_counts().max()) if variable in observations else int(joined[variable].value_counts().max()),
        "data_status": "synthetic",
    })

mode_summary = pd.DataFrame(mode_rows)
mode_summary.to_csv(OUTPUT_DIR / "mode_summary.csv", index=False)
display(mode_summary)
print("Continuous simulated metrics are intentionally summarized without a unique mode.")

## IQR-based potential outlier detection

For each numeric metric, potential outliers are values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR. An outlier flag is not a claim that a record is invalid: these records are reviewed as plausible edge cases created within the synthetic simulation.

In [ ]:
outlier_rows = []
for variable in NUMERIC_METRICS:
    series = observations[variable].astype(float)
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    mask = (series < lower_fence) | (series > upper_fence)

    for _, row in observations.loc[mask].iterrows():
        outlier_rows.append({
            "observation_id": row["observation_id"],
            "ticket_id": row["ticket_id"],
            "variable": variable,
            "value": row[variable],
            "lower_fence": lower_fence,
            "upper_fence": upper_fence,
            "data_status": "synthetic",
            "outlier_rule": "1.5 x IQR",
        })

outliers = pd.DataFrame(outlier_rows, columns=[
    "observation_id", "ticket_id", "variable", "value",
    "lower_fence", "upper_fence", "data_status", "outlier_rule",
])
outliers.to_csv(OUTPUT_DIR / "outliers.csv", index=False)
display(outliers if not outliers.empty else Markdown("No potential outliers detected."))
print(f"Potential outlier flags: {len(outliers)}")
print("Review note: flagged values are plausible synthetic edge cases, not data-quality failures.")
print("Saved outputs/outliers.csv")

## Histograms

Each histogram is labeled as synthetic and uses the required units: seconds, simulated tokens, a 0–1 score/proportion, or an integer error count.

In [ ]:
plot_labels = {
    "execution_time_sec": ("Execution time (seconds, simulated)", "execution_time_histogram.png"),
    "token_usage": ("Token usage (simulated tokens)", "token_usage_histogram.png"),
    "accuracy_score": ("Accuracy score (0–1, simulated)", "accuracy_score_histogram.png"),
    "test_pass_rate": ("Test pass rate (0–1, simulated)", "test_pass_rate_histogram.png"),
    "error_count": ("Error count (simulated count)", "error_count_histogram.png"),
}

for variable, (xlabel, filename) in plot_labels.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    bins = 12 if variable != "error_count" else np.arange(-0.5, observations[variable].max() + 1.5, 1)
    ax.hist(observations[variable], bins=bins, color="#2563eb", edgecolor="white", alpha=0.88)
    ax.set_title(f"Synthetic {variable} distribution")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frequency (synthetic observations)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=160)
    plt.show()
    plt.close(fig)

## Boxplots for the numeric metrics

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_METRICS), figsize=(16, 5))
for ax, variable in zip(axes, NUMERIC_METRICS):
    ax.boxplot(
        observations[variable],
        patch_artist=True,
        boxprops={"facecolor": "#93c5fd", "color": "#1d4ed8"},
        medianprops={"color": "#b91c1c", "linewidth": 2},
        whiskerprops={"color": "#1d4ed8"},
        capprops={"color": "#1d4ed8"},
        flierprops={"marker": "o", "markerfacecolor": "#dc2626", "markersize": 4},
    )
    ax.set_title(variable.replace("_", "\n"))
    ax.set_ylabel("Simulated value")
    ax.grid(axis="y", alpha=0.35)
fig.suptitle("Synthetic numeric metric boxplots", y=1.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "numeric_metrics_boxplots.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## Scatter plots and exploratory association

The plots below show association within this synthetic sample. They do not establish causation, predictive validity, or real AI-agent performance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for difficulty, group in joined.groupby("difficulty", sort=True):
    ax.scatter(
        group["execution_time_sec"],
        group["token_usage"],
        alpha=0.72,
        s=32,
        label=difficulty,
    )
ax.set_title("Synthetic execution time versus token usage")
ax.set_xlabel("Execution time (seconds, simulated)")
ax.set_ylabel("Token usage (simulated tokens)")
ax.legend(title="Difficulty")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "time_vs_tokens_scatter.png", dpi=160)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for difficulty, group in joined.groupby("difficulty", sort=True):
    ax.scatter(
        group["accuracy_score"],
        group["test_pass_rate"],
        alpha=0.72,
        s=32,
        label=difficulty,
    )
ax.set_title("Synthetic accuracy score versus test pass rate")
ax.set_xlabel("Accuracy score (0–1, simulated)")
ax.set_ylabel("Test pass rate (0–1, simulated)")
ax.set_xlim(0.55, 1.01)
ax.set_ylim(0.50, 1.01)
ax.legend(title="Difficulty")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "accuracy_vs_test_pass_scatter.png", dpi=160)
plt.show()
plt.close(fig)

## Correlation matrix

Pearson correlation is used as an exploratory summary of linear association among the synthetic numeric variables. It must not be interpreted as causal evidence or a benchmark claim.

In [ ]:
correlation = observations[NUMERIC_METRICS].corr()
display(correlation.round(3))

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(NUMERIC_METRICS)), labels=NUMERIC_METRICS, rotation=45, ha="right")
ax.set_yticks(range(len(NUMERIC_METRICS)), labels=NUMERIC_METRICS)
for row in range(len(NUMERIC_METRICS)):
    for column in range(len(NUMERIC_METRICS)):
        ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center", color="black")
ax.set_title("Correlation matrix — synthetic metrics")
fig.colorbar(image, ax=ax, label="Pearson correlation (synthetic sample)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "correlation_matrix.png", dpi=160)
plt.show()
plt.close(fig)

## Optional grouped descriptive view

This grouped table uses the joined task metadata to compare the synthetic sample by difficulty. It remains descriptive: no future values are predicted and no machine-learning model is trained.

In [ ]:
difficulty_summary = (
    joined.groupby("difficulty", observed=True)[NUMERIC_METRICS]
    .agg(["count", "mean", "median"])
    .round(3)
)
display(difficulty_summary)

## Final interpretation

The final interpretation is intentionally limited to this synthetic teaching sample. The execution metrics are not real AI-agent or production measurements.

In [ ]:
summary_indexed = descriptive_summary.set_index("variable")
time_median = summary_indexed.loc["execution_time_sec", "median"]
token_median = summary_indexed.loc["token_usage", "median"]
accuracy_median = summary_indexed.loc["accuracy_score", "median"]
pass_median = summary_indexed.loc["test_pass_rate", "median"]
error_median = summary_indexed.loc["error_count", "median"]

upper_triangle = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
strongest_pair = upper_triangle.abs().stack().idxmax()
strongest_value = upper_triangle.loc[strongest_pair]

print("Synthetic-data conclusion")
print("-------------------------")
print(f"Center: median execution time = {time_median:.1f} seconds; median token usage = {token_median:.0f} simulated tokens.")
print(f"Quality center: median accuracy score = {accuracy_median:.3f}; median test pass rate = {pass_median:.3f}; median error count = {error_median:.0f}.")
print(f"Spread: the IQR summaries and boxplots show the distribution and the deliberately reviewable synthetic edge cases.")
print(f"Potential outlier flags: {len(outliers)} records/metrics under the 1.5 x IQR rule.")
print(f"Exploratory association: the strongest absolute Pearson correlation in this synthetic sample is {strongest_pair[0]} vs {strongest_pair[1]} (r = {strongest_value:.3f}).")
print()
print("Limitations: execution_time_sec, token_usage, accuracy_score, test_pass_rate, and error_count are simulated variables.")
print("They are not real Codex runs, real AI-agent performance, production Android measurements, or a validated benchmark.")
print("No forecasting, machine learning, causal inference, or generalization beyond this synthetic sample was performed.")